[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/11-ai-automation/notebooks/03-n8n-workflow-json.ipynb)

# Notebook 03 — Reading, validating and simulating n8n workflows

**Goal:** understand the JSON inside an n8n workflow file well enough to read, check and debug it — without running n8n.

1. Load the three workflows from `../workflows/` (downloaded from GitHub if you're in Colab).
2. List their nodes and connections.
3. Validate the structure (the same checks as `code/validate_workflows.py`).
4. **Simulate** workflow 01 in Python by walking its connections, including the IF branch.

## Step 1 — Load the files

Locally the files are next door in `../workflows`. In Colab they are fetched from the course repo on GitHub.

In [1]:
import json, pathlib, urllib.request

FILES = ["01-webhook-ticket-triage.json", "02-rss-digest-to-sheets.json", "03-rag-answer-webhook.json"]
RAW = "https://raw.githubusercontent.com/vvvvvivekkk/AI-Engineer/main/11-ai-automation/workflows/"
local = pathlib.Path("../workflows")
workflows = {}
for f in FILES:
    p = local / f
    if p.exists():
        workflows[f] = json.loads(p.read_text(encoding="utf-8"))
    else:
        with urllib.request.urlopen(RAW + f) as r:
            workflows[f] = json.loads(r.read().decode())
for f, wf in workflows.items():
    print(f"{f}: '{wf['name']}' - {len(wf['nodes'])} nodes")

01-webhook-ticket-triage.json: '01 Support ticket triage (webhook -> LLM classify -> Slack / email)' - 6 nodes
02-rss-digest-to-sheets.json: '02 Daily RSS digest (schedule -> RSS -> LLM summary -> Google Sheet)' - 6 nodes
03-rag-answer-webhook.json: '03 RAG answer bot (webhook -> search -> LLM -> respond)' - 6 nodes


## Step 2 — What is inside a node?

Each node has a **name** (used by connections and expressions), a **type** (which kind of node), a **typeVersion**, a **position** on the canvas, its **parameters**, and sometimes **credentials** (a reference by id and name — never the secret itself).

In [2]:
wf1 = workflows[FILES[0]]
print(json.dumps(wf1["nodes"][0], indent=2))

{
  "parameters": {
    "httpMethod": "POST",
    "path": "support-ticket",
    "responseMode": "onReceived",
    "options": {}
  },
  "id": "80ff537c-900e-4faf-aa91-b00742d32b72",
  "name": "Webhook",
  "type": "n8n-nodes-base.webhook",
  "typeVersion": 2,
  "position": [
    0,
    300
  ],
  "webhookId": "01da109b-c0d9-43d0-9516-71c8374a63a2"
}


A table of every node in all three workflows, with pandas:

In [3]:
import pandas as pd
rows = []
for f, wf in workflows.items():
    for n in wf["nodes"]:
        rows.append({"workflow": f[:2], "name": n["name"], "type": n["type"].split(".")[-1],
                     "version": n["typeVersion"], "needs credential": ", ".join(n.get("credentials", {})) or "-"})
pd.DataFrame(rows)

,workflow,name,type,version,needs credential
0,01,Webhook,webhook,2.0,-
1,01,Classify with LLM,httpRequest,4.2,httpHeaderAuth
2,01,Parse category,set,3.4,-
3,01,Is urgent?,if,2.0,-
4,01,Alert Slack,slack,2.2,slackApi
5,01,Email support team,emailSend,2.1,smtp
6,02,Every morning 8am,scheduleTrigger,1.2,-
7,02,Read RSS feed,rssFeedRead,1.1,-
8,02,Keep first 5,limit,1.0,-
9,02,Summarise with LLM,httpRequest,4.2,httpHeaderAuth


## Step 3 — Connections

`connections` is keyed by the **source node's name**. `main` is a list of **outputs**; each output is a list of target nodes. An IF node has two outputs: index 0 = true, index 1 = false.

In [4]:
def show_connections(wf):
    for src, outs in wf["connections"].items():
        for i, branch in enumerate(outs["main"]):
            for link in branch:
                label = f" (output {i})" if len(outs["main"]) > 1 else ""
                print(f"  {src}{label} -> {link['node']}")

for f, wf in workflows.items():
    print(f); show_connections(wf)

01-webhook-ticket-triage.json
  Webhook -> Classify with LLM
  Classify with LLM -> Parse category
  Parse category -> Is urgent?
  Is urgent? (output 0) -> Alert Slack
  Is urgent? (output 1) -> Email support team
02-rss-digest-to-sheets.json
  Every morning 8am -> Read RSS feed
  Read RSS feed -> Keep first 5
  Keep first 5 -> Summarise with LLM
  Summarise with LLM -> Shape row
  Shape row -> Append to Google Sheet
03-rag-answer-webhook.json
  Question webhook -> Search knowledge base
  Search knowledge base -> Found context?
  Found context? (output 0) -> Answer with LLM
  Found context? (output 1) -> Reply no answer
  Answer with LLM -> Reply with answer


## Step 4 — Validate

Common reasons an import fails or a workflow silently breaks: missing keys, duplicate names, connections to nodes that don't exist, no trigger, an unreachable node, or a pasted secret. Let's check for all of them.

In [5]:
import re
NODE_KEYS = {"id", "name", "type", "typeVersion", "position", "parameters"}
SECRET_RE = re.compile(r"(sk-[A-Za-z0-9_\-]{20,}|xox[bp]-[A-Za-z0-9-]{10,})")
is_trigger = lambda t: t.endswith("Trigger") or t == "n8n-nodes-base.webhook"

def validate(wf):
    errs = [f"missing '{k}'" for k in ("name", "nodes", "connections") if k not in wf]
    if errs: return errs
    names = [n["name"] for n in wf["nodes"]]
    if len(set(names)) != len(names): errs.append("duplicate node names")
    for n in wf["nodes"]:
        if NODE_KEYS - n.keys(): errs.append(f"{n['name']}: missing {NODE_KEYS - n.keys()}")
    if not any(is_trigger(n["type"]) for n in wf["nodes"]): errs.append("no trigger node")
    targets = {l["node"] for o in wf["connections"].values() for b in o["main"] for l in b}
    errs += [f"unknown target {t}" for t in targets - set(names)]
    errs += [f"unreachable node {n['name']}" for n in wf["nodes"] if not is_trigger(n["type"]) and n["name"] not in targets]
    if SECRET_RE.search(json.dumps(wf)): errs.append("secret-looking string found")
    return errs

for f, wf in workflows.items():
    print(f, "->", validate(wf) or "OK")

01-webhook-ticket-triage.json -> OK
02-rss-digest-to-sheets.json -> OK
03-rag-answer-webhook.json -> OK


Now break a copy on purpose and watch the validator catch it — this is how you'd debug a hand-edited file.

In [6]:
import copy
broken = copy.deepcopy(wf1)
broken["nodes"][1]["name"] = "Classify (renamed)"      # renamed, but connections still use the old name
broken["nodes"][2]["parameters"]["note"] = "sk-" + "a" * 30   # someone pasted a fake key
print(validate(broken))

['unknown target Classify with LLM', 'unreachable node Classify (renamed)', 'secret-looking string found']


Renaming a node in the n8n editor updates connections and expressions for you; editing the JSON by hand does not. That is why the validator matters.

## Step 5 — Simulate workflow 01

n8n expressions are JavaScript, so we won't run them. Instead each node **type** gets a small Python stand-in, and a little engine follows the `connections` from the trigger, exactly as n8n does: run a node, pick the output (IF nodes choose 0 or 1), pass the item to the next node.

In [7]:
def fake_llm_classify(subject, message):
    text = (subject + " " + message).lower()
    cat = "urgent" if any(w in text for w in ["down", "error", "urgent"]) else "billing" if "refund" in text else "general"
    return {"choices": [{"message": {"content": json.dumps({"category": cat, "reason": "keyword match (offline stand-in)"})}}]}

def run_node(node, item, history):
    t = node["type"].split(".")[-1]
    if t == "webhook":
        return item, 0
    if t == "httpRequest":                                   # the LLM call
        body = history["Webhook"]["body"]
        return fake_llm_classify(body["subject"], body["message"]), 0
    if t == "set":                                           # Parse category
        parsed = json.loads(item["choices"][0]["message"]["content"])
        return {**parsed, "email": history["Webhook"]["body"]["email"], "subject": history["Webhook"]["body"]["subject"]}, 0
    if t == "if":
        cond = node["parameters"]["conditions"]["conditions"][0]
        return item, 0 if item["category"] == cond["rightValue"] else 1
    if t == "slack":
        print(f"    [SLACK {node['parameters']['channelId']['value']}] URGENT from {item['email']}: {item['subject']}")
        return item, 0
    if t == "emailSend":
        print(f"    [EMAIL to {node['parameters']['toEmail']}] [{item['category']}] {item['subject']}")
        return item, 0
    raise ValueError(f"no stand-in for {t}")

def simulate(wf, trigger_item):
    nodes = {n["name"]: n for n in wf["nodes"]}
    current = next(n["name"] for n in wf["nodes"] if is_trigger(n["type"]))
    item, history = trigger_item, {}
    while current:
        item, out = run_node(nodes[current], item, history)
        history[current] = item
        print(f"  ran {current!r:24} -> output {out}")
        branches = wf["connections"].get(current, {}).get("main", [])
        current = branches[out][0]["node"] if out < len(branches) and branches[out] else None
    return history

for subject, message in [("Site is down", "Error 500 on checkout, urgent!"), ("Question", "Do you have gift cards?")]:
    print(f"\nTicket: {subject}")
    simulate(wf1, {"body": {"email": "asha@example.com", "subject": subject, "message": message}})


Ticket: Site is down
  ran 'Webhook'                -> output 0
  ran 'Classify with LLM'      -> output 0
  ran 'Parse category'         -> output 0
  ran 'Is urgent?'             -> output 0
    [SLACK #support-urgent] URGENT from asha@example.com: Site is down
  ran 'Alert Slack'            -> output 0

Ticket: Question
  ran 'Webhook'                -> output 0
  ran 'Classify with LLM'      -> output 0
  ran 'Parse category'         -> output 0
  ran 'Is urgent?'             -> output 1
    [EMAIL to support@example.com] [general] Question
  ran 'Email support team'     -> output 0


The urgent ticket took output 0 of the IF node (Slack); the other took output 1 (email) — just like n8n's execution view would show.

## Step 6 — Look at the LLM request a node would send

The `jsonBody` parameter is an n8n **expression** (it starts with `=` and contains `{{ }}`). Reading it tells you exactly what will be sent to the model:

In [8]:
llm_node = next(n for n in wf1["nodes"] if n["name"] == "Classify with LLM")
print("URL:   ", llm_node["parameters"]["url"])
print("Auth:  ", llm_node["parameters"]["genericAuthType"], "->", llm_node["credentials"])
print("Body expression:\n", llm_node["parameters"]["jsonBody"][:400], "...")

URL:    https://api.openai.com/v1/chat/completions
Auth:   httpHeaderAuth -> {'httpHeaderAuth': {'id': 'REPLACE_WITH_YOUR_CREDENTIAL_ID', 'name': 'OpenAI header auth (Authorization: Bearer ...)'}}
Body expression:
 ={{ JSON.stringify({ model: 'gpt-5-mini', response_format: { type: 'json_object' }, messages: [ { role: 'system', content: 'You classify customer support tickets. Reply ONLY with JSON like {"category": "urgent", "reason": "short reason"}. category must be one of: urgent, billing, general.' }, { role: 'user', content: 'Subject: ' + $json.body.subject + '\nMessage: ' + $json.body.message } ] }) }} ...


## Recap

* A workflow file = `name` + `nodes` + `connections` (keyed by source **name**).
* IF nodes have two outputs (0 true, 1 false).
* Credentials are referenced, never embedded — keep it that way before sharing files.
* A tiny validator catches most hand-editing mistakes.

**Try it:** extend `run_node` and `simulate` for workflow 03 (stand-ins for the search request and `respondToWebhook`) and run one question that has an answer and one that doesn't.